In [1]:
from DWG_to_Excel_ETL import *
import os
import shutil
import numpy as np
from scipy.spatial.distance import cdist

dwg_folder = r"C:\Users\Usuario\Ecotechnee\Office - Documentos\Projetos\379 - Zagope - UFV Capela\01 - Projetos\01 - Em Confecção\02 - Edição\01 - PV\01 - Layout"
temp_folder = r"C:\Users\Usuario\Ecotechnee\Office - Documentos\Projetos\379 - Zagope - UFV Capela\01 - Projetos\01 - Em Confecção\02 - Edição\01 - PV\01 - Layout\temp"
output_folder = r"C:\Users\Usuario\Ecotechnee\Office - Documentos\Projetos\379 - Zagope - UFV Capela\01 - Projetos\01 - Em Confecção\02 - Edição\01 - PV\04 - Base de Dados"

os.makedirs(temp_folder, exist_ok=True)
dwg_file1 = os.path.join(dwg_folder, "PV Capela - Rede de Baixa Tensão.dwg")
dwg_file2 = os.path.join(dwg_folder, "PV Capela - Eletromecânico.dwg")

shutil.copy(dwg_file1, temp_folder)
shutil.copy(dwg_file2, temp_folder)

convert_dwg_to_dxf(temp_folder, output_folder)

dxf_file1 = os.path.join(output_folder, "PV Capela - Rede de Baixa Tensão.dxf")
dxf_file2 = os.path.join(output_folder, "PV Capela - Eletromecânico.dxf")

data_LV = extract_data_from_dxf(dxf_file1)
print("LV Data Loaded")
data_ELM = extract_data_from_dxf(dxf_file2)
print("ELM Data Loaded")

os.remove(dxf_file1)
os.remove(dxf_file2)

LV Data Loaded
ELM Data Loaded


In [2]:
data_LV_df = pd.DataFrame(data_LV)
data_LV_df.drop(columns=["type", "handle"], inplace=True)
print(data_LV_df.describe())
print(data_LV_df.info())
print(data_LV_df)

            start_x        start_y         end_x          end_y       length
count   4918.000000    4918.000000   4918.000000    4918.000000  4918.000000
mean  -18656.745720  157466.554531 -18650.659525  157466.648957    70.991977
std      450.203409     600.092684    438.613461     599.973403    72.331042
min   -19678.152000  156352.961300 -19624.782000  156353.768900     0.182400
25%   -19001.494900  156884.461300 -18987.193400  156887.384200    17.275522
50%   -18700.970950  157623.283900 -18700.434900  157623.884200    47.849193
75%   -18270.048500  157988.783900 -18253.323000  157989.384200    95.972825
max   -17787.399300  158305.783900 -17790.617700  158297.622600   394.804545
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4918 entries, 0 to 4917
Data columns (total 6 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   layer    4918 non-null   object 
 1   start_x  4918 non-null   float64
 2   start_y  4918 non-null   float64
 3   end_x 

In [3]:
data_ELM_df = pd.DataFrame(data_ELM)
data_ELM_df.drop(columns=["type", "handle", "rotation"], inplace=True)
print(data_ELM_df.describe())
print(data_ELM_df.info())
print(data_ELM_df)

            start_x        start_y
count   4476.000000    4476.000000
mean  -18651.871093  157476.304707
std      456.840158     595.032941
min   -19698.300000  156352.961300
25%   -19008.387100  156923.041950
50%   -18670.111700  157634.283900
75%   -18277.855650  157986.783900
max   -17791.577226  158320.185535
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4476 entries, 0 to 4475
Data columns (total 14 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   layer    4476 non-null   object 
 1   name     4476 non-null   object 
 2   start_x  4476 non-null   float64
 3   start_y  4476 non-null   float64
 4   PRK      4476 non-null   object 
 5   UFV      4476 non-null   object 
 6   PST      4476 non-null   object 
 7   INV      4466 non-null   object 
 8   QBT      186 non-null    object 
 9   DSJ      186 non-null    object 
 10  POT      4466 non-null   object 
 11  GRP      4466 non-null   object 
 12  EFX      4280 non-null   object 
 13  STR

In [4]:
mask = (data_LV_df['layer'] == "EMF_Capela_I_em_vala_DC") | (data_LV_df['layer'] == "EMF_Capela_II_em_vala_DC")

vala_solar_df = data_LV_df[mask].copy()
print(vala_solar_df)

                        layer       start_x        start_y       end_x  \
0     EMF_Capela_I_em_vala_DC -19019.984400  156505.268900 -19018.9999   
1     EMF_Capela_I_em_vala_DC -19019.984400  156497.268900 -19018.9999   
2     EMF_Capela_I_em_vala_DC -19019.984400  156489.268900 -19018.9999   
3     EMF_Capela_I_em_vala_DC -19019.984400  156481.268900 -19018.9999   
4     EMF_Capela_I_em_vala_DC -19019.984400  156473.268900 -19018.9999   
..                        ...           ...            ...         ...   
633  EMF_Capela_II_em_vala_DC -18946.402600  158017.268900 -18945.5431   
634  EMF_Capela_II_em_vala_DC -18945.543100  158033.268900 -18945.4431   
635  EMF_Capela_II_em_vala_DC -18899.171464  157921.248815 -18902.8471   
636  EMF_Capela_II_em_vala_DC -18898.234600  157913.268900 -18897.3751   
637   EMF_Capela_I_em_vala_DC -18752.139000  157189.089136 -18751.2795   

           end_y     length  
0    156505.2689   0.984500  
1    156497.2689   0.984500  
2    156489.2689   0.

In [5]:
mask = (data_ELM_df['name'] == "EMF379_string_1") | (data_ELM_df['name'] == "EMF379_string_2")

strings_df = data_ELM_df[mask].copy()
print(strings_df.describe())
print(strings_df.info())
print(strings_df)

            start_x        start_y
count   4280.000000    4280.000000
mean  -18651.921753  157476.036202
std      457.577648     594.942327
min   -19698.300000  156352.961300
25%   -19008.387100  156923.041950
50%   -18667.613500  157632.541950
75%   -18280.373350  157986.783900
max   -17796.983300  158305.783900
<class 'pandas.core.frame.DataFrame'>
Index: 4280 entries, 44 to 4323
Data columns (total 14 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   layer    4280 non-null   object 
 1   name     4280 non-null   object 
 2   start_x  4280 non-null   float64
 3   start_y  4280 non-null   float64
 4   PRK      4280 non-null   object 
 5   UFV      4280 non-null   object 
 6   PST      4280 non-null   object 
 7   INV      4280 non-null   object 
 8   QBT      0 non-null      object 
 9   DSJ      0 non-null      object 
 10  POT      4280 non-null   object 
 11  GRP      4280 non-null   object 
 12  EFX      4280 non-null   object 
 13  STR    

In [6]:
mask = (data_ELM_df['name'] == "EMF379_inversor")

inversor_df = data_ELM_df[mask].copy()
inversor_df['GRP'] = inversor_df["PRK"].astype(str) + "." + inversor_df["UFV"].astype(str) + "." + inversor_df["PST"].astype(str) + "." + inversor_df["INV"].astype(str)
inversor_df.rename(columns={'start_x':'inverter_x', 'start_y':'inverter_y'}, inplace=True)
inversor_df.drop(columns=['layer','name', 'PRK', 'UFV', 'PST', 'INV', 'QBT', 'DSJ', 'POT', 'EFX', 'STR'], inplace=True)
print(inversor_df.describe())
print(inversor_df.info())
print(inversor_df)

         inverter_x     inverter_y
count    186.000000     186.000000
mean  -18646.675095  157477.816133
std      441.137493     599.073845
min   -19593.644494  156380.884204
25%   -18985.029168  156919.384204
50%   -18700.434895  157637.384204
75%   -18248.307363  157993.876762
max   -17791.577226  158290.384204
<class 'pandas.core.frame.DataFrame'>
Index: 186 entries, 0 to 4465
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   inverter_x  186 non-null    float64
 1   inverter_y  186 non-null    float64
 2   GRP         186 non-null    object 
dtypes: float64(2), object(1)
memory usage: 5.8+ KB
None
        inverter_x     inverter_y       GRP
0    -19494.887410  156646.455636   1.2.2.1
1    -19511.253312  156614.455636   1.2.2.2
2    -19550.948494  156582.455636   1.2.2.3
3    -19593.644494  156558.455636   1.2.2.5
4    -19357.238587  156504.384204   1.2.2.8
...            ...            ...       ...
4461 -18690.903

In [7]:
for idx, row in strings_df.iterrows():
    x = row['start_x']
    y = row['start_y']

    mask = (
        (strings_df['start_y'] >= y - 5) &
        (strings_df['start_y'] <= y + 5) &
        (strings_df['start_x'] >= x - 0.5) &
        (strings_df['start_x'] <= x + 0.5) &
        (strings_df.index != idx)
    )

    if mask.any():
        y_neighbors = strings_df.loc[mask, 'start_y']

        if (y > y_neighbors).any():
            strings_df.loc[idx, 'Type'] = "T"
        else:
            strings_df.loc[idx, 'Type'] = "B"

    else:
        strings_df.loc[idx, 'Type'] = "C"


strings_df.loc[:,'Line'] = np.where(
    strings_df["Type"] == "C",
    round(strings_df["start_y"]),
    np.where(
        strings_df["Type"] == "T",
        round(strings_df["start_y"] - 1.1613),
        round(strings_df["start_y"] + 1.1613)
    )
)


strings_df["PRK.UFV.PST.INV"] = strings_df["PRK"].astype(str) + "." + strings_df["UFV"].astype(str) + "." + strings_df["PST"].astype(str) + "." + strings_df["INV"].astype(str)

strings_df.sort_values(by=["Line", "start_x"], ascending=[False,True], inplace=True, ignore_index=True)

In [8]:
#Different Lines
condition1 = strings_df["Line"] != strings_df["Line"].shift()

#Break in the Structure with no C type on the edges
condition2 = ((abs(strings_df["start_x"] - strings_df["start_x"].shift()) > 45) &
              (strings_df["Type"] != "C") &
              (strings_df["Type"].shift() != "C"))

#Break in the Structure with one C type on the edges
condition3 = ((abs(strings_df["start_x"] - strings_df["start_x"].shift()) > 35) &
              ((strings_df["Type"] == "C") != (strings_df["Type"].shift() == "C")))

#Break in the Structure with two C types on the edges
condition4 = ((abs(strings_df["start_x"] - strings_df["start_x"].shift()) > 22) &
              (strings_df["Type"] == "C") &
              (strings_df["Type"].shift() == "C"))

#Different PSTs
condition5 = strings_df["PRK.UFV.PST.INV"] != strings_df["PRK.UFV.PST.INV"].shift()




break_condition_1 = condition1 | condition2 | condition3 | condition4 | condition5
break_condition_2 = condition1 | condition2 | condition3 | condition4

strings_df['SubGroup'] = break_condition_1.cumsum()
strings_df['Group'] = break_condition_2.cumsum()
print(strings_df)

                  layer             name     start_x      start_y PRK UFV PST  \
0     EMF_em_MVPS_2.1.2  EMF379_string_1 -18349.6460  158304.6226   2   1   2   
1     EMF_em_MVPS_2.1.2  EMF379_string_2 -18317.6140  158305.7839   2   1   2   
2     EMF_em_MVPS_2.1.2  EMF379_string_2 -18317.6140  158303.4613   2   1   2   
3     EMF_em_MVPS_2.1.2  EMF379_string_2 -18274.9180  158305.7839   2   1   2   
4     EMF_em_MVPS_2.1.2  EMF379_string_2 -18274.9180  158303.4613   2   1   2   
...                 ...              ...         ...          ...  ..  ..  ..   
4275  EMF_em_MVPS_1.3.1  EMF379_string_2 -19107.1669  156363.2839   1   3   1   
4276  EMF_em_MVPS_1.3.1  EMF379_string_2 -19107.1669  156360.9613   1   3   1   
4277  EMF_em_MVPS_1.3.1  EMF379_string_1 -19075.1549  156362.1226   1   3   1   
4278  EMF_em_MVPS_1.3.1  EMF379_string_2 -19096.4929  156355.2839   1   3   1   
4279  EMF_em_MVPS_1.3.1  EMF379_string_2 -19096.4929  156352.9613   1   3   1   

     INV  QBT  DSJ  POT    

In [9]:
strings_df['W exit'] = np.where(
    strings_df["Type"] == "C",
    strings_df['start_x'] - 9.9325,
    strings_df['start_x'] - 20.4965
)

strings_df['E exit'] = np.where(
    strings_df["Type"] == "C",
    strings_df['start_x'] + 9.9325,
    strings_df['start_x'] + 20.4965
)

strings_df['SubGroup - Position X - W'] = strings_df.groupby("SubGroup")["W exit"].transform('min')
strings_df['SubGroup - Position X - E'] = strings_df.groupby("SubGroup")["E exit"].transform('max')

strings_df['Group - Position X - W'] = strings_df.groupby("Group")["W exit"].transform('min')
strings_df['Group - Position X - E'] = strings_df.groupby("Group")["E exit"].transform('max')

strings_df['No. of Strings'] = strings_df.groupby("SubGroup")['name'].transform('count')

print(strings_df)

                  layer             name     start_x      start_y PRK UFV PST  \
0     EMF_em_MVPS_2.1.2  EMF379_string_1 -18349.6460  158304.6226   2   1   2   
1     EMF_em_MVPS_2.1.2  EMF379_string_2 -18317.6140  158305.7839   2   1   2   
2     EMF_em_MVPS_2.1.2  EMF379_string_2 -18317.6140  158303.4613   2   1   2   
3     EMF_em_MVPS_2.1.2  EMF379_string_2 -18274.9180  158305.7839   2   1   2   
4     EMF_em_MVPS_2.1.2  EMF379_string_2 -18274.9180  158303.4613   2   1   2   
...                 ...              ...         ...          ...  ..  ..  ..   
4275  EMF_em_MVPS_1.3.1  EMF379_string_2 -19107.1669  156363.2839   1   3   1   
4276  EMF_em_MVPS_1.3.1  EMF379_string_2 -19107.1669  156360.9613   1   3   1   
4277  EMF_em_MVPS_1.3.1  EMF379_string_1 -19075.1549  156362.1226   1   3   1   
4278  EMF_em_MVPS_1.3.1  EMF379_string_2 -19096.4929  156355.2839   1   3   1   
4279  EMF_em_MVPS_1.3.1  EMF379_string_2 -19096.4929  156352.9613   1   3   1   

     INV  QBT  DSJ  ... PRK

In [10]:
groups_df = strings_df.groupby("SubGroup")[['PRK.UFV.PST.INV','Line', 'SubGroup - Position X - W', 'SubGroup - Position X - E', 'Group - Position X - W', 'Group - Position X - E', 'No. of Strings']].first()
groups_df.rename(columns={'PRK.UFV.PST.INV': 'GRP', 'Line': 'Y', 'Group - Position X - W': 'X-W-STRUCT', 'Group - Position X - E': 'X-E-STRUCT', 'SubGroup - Position X - W': 'X-W-STR', 'SubGroup - Position X - E': 'X-E-STR'}, inplace=True)
print(f'Maximum number of strings in a group: {groups_df['No. of Strings'].max()}\n')
print(groups_df)

Maximum number of strings in a group: 20

               GRP         Y     X-W-STR     X-E-STR  X-W-STRUCT  X-E-STRUCT  \
SubGroup                                                                       
1          2.1.2.1  158305.0 -18359.5785 -18254.4215 -18359.5785 -18254.4215   
2          2.2.2.1  158303.0 -18757.6950 -18737.8300 -18757.6950 -18737.8300   
3          2.1.2.2  158298.0 -18404.2380 -18384.3730 -18404.2380 -18384.3730   
4          2.1.2.1  158297.0 -18372.8685 -18246.4835 -18372.8685 -18246.4835   
5          2.2.2.1  158295.0 -18757.6950 -18737.8300 -18757.6950 -18737.8300   
...            ...       ...         ...         ...         ...         ...   
847       1.3.1.15  156387.0 -19170.3594 -19065.2224 -19170.3594 -19065.2224   
848       1.3.1.15  156379.0 -19149.0114 -19065.3224 -19149.0114 -19065.3224   
849       1.3.1.15  156371.0 -19127.6634 -19065.2224 -19127.6634 -19065.2224   
850       1.3.1.15  156362.0 -19127.6634 -19065.2224 -19127.6634 -19065.2224  

Duct_map is needed to represent how many ducts are necessary and how many strings are there in each duct. Given that the maximum of strings in a duct is 3, the duct_map is as follows:
=

In [11]:
duct_map = {
    1: '-1-',
    2: '-2-',
    3: '-3-',
    4: '-3-1-',
    5: '-3-2-',
    6: '-3-3-',
    7: '-3-3-1-',
    8: '-3-3-2-',
    9: '-3-3-3-',
    10: '-3-3-3-1-',
    11: '-3-3-3-2-',
    12: '-3-3-3-3-',
    13: '-3-3-3-3-1-',
    14: '-3-3-3-3-2-',
    15: '-3-3-3-3-3-',
    16: '-3-3-3-3-3-1-',
    17: '-3-3-3-3-3-2-',
    18: '-3-3-3-3-3-3-',
    19: '-3-3-3-3-3-3-1-',
    20: '-3-3-3-3-3-3-2-',
}

groups_df.loc[:,'Duct Definition'] = groups_df['No. of Strings'].map(duct_map)
groups_df['Number of Ducts'] = groups_df['Duct Definition'].astype(str).str.count('-')-1
groups_df = groups_df.merge(inversor_df, on='GRP')
groups_df['W or E'] = np.where(
    abs(groups_df['X-W-STRUCT'] - groups_df['inverter_x']) > abs(groups_df['X-E-STRUCT'] - groups_df['inverter_x']),
    "E",
    "W"
)

groups_df['closest_x'] = np.where(
    groups_df['W or E'] == "W",
    groups_df['X-W-STRUCT'],
    groups_df['X-E-STRUCT']
)
print(groups_df)

          GRP         Y     X-W-STR     X-E-STR  X-W-STRUCT  X-E-STRUCT  \
0     2.1.2.1  158305.0 -18359.5785 -18254.4215 -18359.5785 -18254.4215   
1     2.2.2.1  158303.0 -18757.6950 -18737.8300 -18757.6950 -18737.8300   
2     2.1.2.2  158298.0 -18404.2380 -18384.3730 -18404.2380 -18384.3730   
3     2.1.2.1  158297.0 -18372.8685 -18246.4835 -18372.8685 -18246.4835   
4     2.2.2.1  158295.0 -18757.6950 -18737.8300 -18757.6950 -18737.8300   
..        ...       ...         ...         ...         ...         ...   
846  1.3.1.15  156387.0 -19170.3594 -19065.2224 -19170.3594 -19065.2224   
847  1.3.1.15  156379.0 -19149.0114 -19065.3224 -19149.0114 -19065.3224   
848  1.3.1.15  156371.0 -19127.6634 -19065.2224 -19127.6634 -19065.2224   
849  1.3.1.15  156362.0 -19127.6634 -19065.2224 -19127.6634 -19065.2224   
850  1.3.1.15  156354.0 -19116.9894 -19075.9964 -19116.9894 -19075.9964   

     No. of Strings Duct Definition  Number of Ducts    inverter_x  \
0                 5          

Delta X Delta Y on the structure coordinates and the inverter and creating the final Dataframe for the Duct Script
=

In [12]:
script_df = groups_df[['No. of Strings', 'Duct Definition', 'Number of Ducts', 'closest_x', 'Y', 'inverter_x', 'inverter_y']].copy()
script_df.rename(columns={'closest_x': 'X'}, inplace=True)
script_df['intermediary_y'] = np.where(
    script_df['inverter_y'] > script_df['Y'],
    script_df['inverter_y'] - 1.3,
    script_df['inverter_y'] + 1.3
)

script_df['inverter_y'] = np.where(
    script_df['inverter_y'] > script_df['Y'],
    script_df['inverter_y'] - 0.1,
    script_df['inverter_y'] + 0.1
)

script_df['Script Valas'] = script_df.apply(
    lambda row: f"_.pline {row['X']},{row['Y']} {row['inverter_x']},{row['Y']} {row['inverter_x']},{row['intermediary_y']} {row['inverter_x']},{row['inverter_y']} ",
    axis=1
)
print(script_df['Script Valas'])

with open (r"C:\Users\Usuario\Desktop\Pessoal\Python\Ecotechne\Dutos DC\Script Valas.scr", 'w') as f:
    f.write('\n'.join(script_df["Script Valas"].astype(str)) + '\n')

script_df['Script Dutos 1'] = script_df.apply(
    lambda row: f"_.pline {row['X']},{row['Y']} {row['inverter_x']},{row['Y']} {row['inverter_x']},{row['intermediary_y']} {row['inverter_x']},{row['inverter_y']} " * str(row['Duct Definition']).count('1'),
    axis=1
)
script_df['Script Dutos 2'] = script_df.apply(
    lambda row: f"_.pline {row['X']},{row['Y']} {row['inverter_x']},{row['Y']} {row['inverter_x']},{row['intermediary_y']} {row['inverter_x']},{row['inverter_y']} " * str(row['Duct Definition']).count('2'),
    axis=1
)
script_df['Script Dutos 3'] = script_df.apply(
    lambda row: (f"_.pline {row['X']},{row['Y']} {row['inverter_x']},{row['Y']} {row['inverter_x']},{row['intermediary_y']} {row['inverter_x']},{row['inverter_y']} " + "\n") * str(row['Duct Definition']).count('3'),
    axis=1
)

with open(r"C:\Users\Usuario\Desktop\Pessoal\Python\Ecotechne\Dutos DC\Script Dutos.scr", 'w') as f:
    # Layer 1
    layer1_commands = script_df["Script Dutos 1"].astype(str)
    layer1_commands = layer1_commands[layer1_commands != '']  # Remove empty strings
    if len(layer1_commands) > 0:  # Only write layer if there are commands
        f.write("LAYER S EMF_bt_eletroduto_1S\n\n")
        f.write('\n'.join(layer1_commands) + '\n')

    # Layer 2
    layer2_commands = script_df["Script Dutos 2"].astype(str)
    layer2_commands = layer2_commands[layer2_commands != '']
    if len(layer2_commands) > 0:
        f.write("LAYER S EMF_bt_eletroduto_2S\n\n")
        f.write('\n'.join(layer2_commands) + '\n')

    # Layer 3
    layer3_commands = script_df["Script Dutos 3"].astype(str)
    layer3_commands = layer3_commands[layer3_commands != '']
    if len(layer3_commands) > 0:
        f.write("LAYER S EMF_bt_eletroduto_3S\n\n")
        f.write(''.join(layer3_commands) + '\n')

0      _.pline -18359.5785,158305.0 -18371.6304851153...
1      _.pline -18757.695,158303.0 -18768.11558725917...
2      _.pline -18384.373,158298.0 -18385.68867078049...
3      _.pline -18372.8685,158297.0 -18371.6304851153...
4      _.pline -18757.695,158295.0 -18768.11558725917...
                             ...                        
846    _.pline -19065.222400000002,156387.0 -19066.55...
847    _.pline -19065.322399999997,156379.0 -19066.55...
848    _.pline -19065.222400000002,156371.0 -19066.55...
849    _.pline -19065.222400000002,156362.0 -19066.55...
850    _.pline -19075.9964,156354.0 -19066.5554084667...
Name: Script Valas, Length: 851, dtype: object


Searching for the closest start- or end-point of the trenches in "vala_solar_df" for each group of strings
=

In [13]:
# start_points = vala_solar_df[['start_x', 'start_y']].values
# end_points = vala_solar_df[['end_x', 'end_y']].values
# west_structure_points = groups_df[['X-W', 'Y']].values
# east_structure_points = groups_df[['X-E', 'Y']].values
#
#
# west_dist_to_start = cdist(west_structure_points, start_points)
# west_dist_to_end = cdist(west_structure_points, end_points)
# east_dist_to_start = cdist(east_structure_points, start_points)
# east_dist_to_end = cdist(east_structure_points, end_points)
#
#
# trench_y = vala_solar_df[['start_y']].values
# structures_y = groups_df[['Y']].values
#
# y_diff = np.abs(structures_y - trench_y.T)
# y_mask = (y_diff <= 5)
#
# dist_to_start = np.minimum(west_dist_to_start, east_dist_to_start)
# dist_to_end = np.minimum(west_dist_to_end, east_dist_to_end)
#
# min_dist_to_trench = np.minimum(
#     dist_to_start,
#     dist_to_end
# )
#
# min_dist_to_trench_masked = np.where(y_mask, min_dist_to_trench, np.inf)
#
# closest_trench_indices = np.argmin(min_dist_to_trench_masked, axis=1)
# closest_distances = np.min(min_dist_to_trench_masked, axis=1)
#
# start_is_closer = dist_to_start[np.arange(len(groups_df)), closest_trench_indices] <= dist_to_end[np.arange(len(groups_df)), closest_trench_indices]
#
# groups_df['closest_trench_id'] = vala_solar_df.index[closest_trench_indices]
# groups_df['distance_to_trench'] = closest_distances
# groups_df['closest_x'] = np.where(
#     start_is_closer,
#     vala_solar_df.iloc[closest_trench_indices]['start_x'].values,
#     vala_solar_df.iloc[closest_trench_indices]['end_x'].values
# )
#
# groups_df['closest_y'] = np.where(
#     start_is_closer,
#     vala_solar_df.iloc[closest_trench_indices]['start_y'].values,
#     vala_solar_df.iloc[closest_trench_indices]['end_y'].values
# )
#
# groups_df = groups_df.merge(inversor_df, on='GRP')
#
#
# groups_df.to_excel(r"C:\Users\Usuario\Desktop\teste.xlsx")

Creating the final Dataframe for the Trench script and the Duct Script
=

In [14]:
# script_df = groups_df[['No. of Strings', 'Duct Definition', 'Number of Ducts', 'closest_x', 'closest_y', 'inverter_x', 'inverter_y']].copy()
# script_df['intermediary_y'] = np.where(
#     script_df['inverter_y'] > script_df['closest_y'],
#     script_df['inverter_y'] - 1.3,
#     script_df['inverter_y'] + 1.3
# )
#
# script_df['inverter_y'] = np.where(
#     script_df['inverter_y'] > script_df['closest_y'],
#     script_df['inverter_y'] - 0.1,
#     script_df['inverter_y'] + 0.1
# )
#
# script_df['Script Valas'] = script_df.apply(
#     lambda row: f"_.pline {row['closest_x']},{row['closest_y']} {row['inverter_x']},{row['closest_y']} {row['inverter_x']},{row['intermediary_y']} {row['inverter_x']},{row['inverter_y']} ",
#     axis=1
# )
# print(script_df['Script Valas'])
#
# with open (r"C:\Users\Usuario\Desktop\Pessoal\Python\Ecotechne\Dutos DC\Script Valas.scr", 'w') as f:
#     f.write('\n'.join(script_df["Script Valas"].astype(str)) + '\n')

DEU RUIM ESSA ÚLTIMA IDEIA, VAMOS PARA A PRÓXIMA:
* CRIAR DOIS GRUPOS, UM SÓ PRA LINHA DE ESTRUTURAS E OUTRO CONSIDERANDO PRA QUAL INVERSOR AS STRINGS ESTÃO INDO, OU SEJA,VAI SER UM SUBGRUPO DESSE PRIMEIRO
* NÃO FAZ SENTIDO O SCRIPT DE VALAS SENDO QUE EU PEGO AS VALAS DO DWG, OU EU TENTO COPIAR A PLANILHA DO HIDEO E FAZER AS VALAS DO ZERO, OU FAÇO UM SIMPLES DELTAX DELTAY DA ESTRUTURA PRO INVERSOR